# Real FAOSTAT run — min-cost-flow for nitrogen

End-to-end pipeline for **one nutrient (N)** and **one year**.

Expected files under `../data/`:
- `production_Nitrogen_N.csv`
- `trade_Nitrogen_N.csv`  (FAOSTAT detailed trade matrix, nutrient nitrogen)

See `../data/README.md` for download instructions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import pandas as pd

from src.preprocessing import load_scenario, apply_supply_shock
from src.model import FertilizerMCF
from src.postprocessing import summary_table, build_comparison, global_summary, save_result

DATA = Path('../data')
YEAR = 2020


## 1. Load the scenario

In [ ]:
scenario = load_scenario(
    production_path=DATA / 'production_Nitrogen_N.csv',
    trade_path=DATA / 'trade_Nitrogen_N.csv',
    year=YEAR,
    cost='inverse_volume',
)
print(f"{len(scenario['countries'])} countries; "
      f"{len(scenario['edges']):,} trade edges")
print(f"Total supply: {scenario['supply'].sum():,.0f} t")
print(f"Total demand: {scenario['demand'].sum():,.0f} t")


## 2. Baseline solve

In [ ]:
baseline = FertilizerMCF(
    supply=scenario['supply'],
    demand=scenario['demand'],
    edges=scenario['edges'],
).solve()

print('feasible :', baseline.feasible)
print('cost     :', round(baseline.total_cost, 0))
print('flow     :', round(baseline.total_flow, 0))


## 3. Apply a supply shock

Example: 60% cut to Russian nitrogen production (keep 40%).
Feel free to edit the `shock` dict below to try other scenarios.

In [ ]:
shock = {'Russian Federation': 0.4}
shocked_supply = apply_supply_shock(scenario['supply'], shock)

shocked = FertilizerMCF(
    supply=shocked_supply,
    demand=scenario['demand'],
    edges=scenario['edges'],
).solve()
print('feasible :', shocked.feasible)


## 4. Compare baseline vs shocked

In [ ]:
global_summary(baseline, shocked)

In [ ]:
comparison = build_comparison(baseline, shocked)
comparison.nsmallest(15, 'change_pp')


## 5. Quick plots

In [ ]:
from src.utils import plot_flow_sankey, plot_coverage_comparison

plot_flow_sankey(baseline, title=f'Baseline flows, {YEAR}',
                 top_k=60).show()
plot_coverage_comparison(comparison,
                         title=f'Coverage drop (N, {YEAR})').show()


## 6. Save

In [ ]:
save_result(baseline, '../results', tag=f'baseline_N_{YEAR}')
save_result(shocked, '../results', tag=f'shock_russia_N_{YEAR}')
